# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jaivarshini-06/flyrank-ml-internship-jaivarshini/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Load the anonymized dataset from the repository
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "Jaivarshini-06/flyrank-ml-internship-jaivarshini/"
    "main/data/raw/content_refresh_anonymized.csv"
)
df = pd.read_csv(DATA_URL)

# Required columns for preparing the feature vector
required = [
    "content_id", "client_id", "impressions_90d",
    "sessions_90d", "content_age_days", "trend_direction"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Define the model features exactly as in scripts/ml_utils.py
numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d",
    "log_ai_sessions_90d", "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct"
]

categorical_features = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier"
]

# Convert numeric values and replace infinities with missing values
for col in numeric_features:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    else:
        df[col] = np.nan

# Fill missing categorical values consistently
for col in categorical_features:
    if col not in df.columns:
        df[col] = "unknown"
    df[col] = df[col].fillna("unknown").astype(str)
    df[col] = df[col].replace({"": "unknown", "nan": "unknown"})

# Create engineered features used by the model
for col in [
    "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"
]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).clip(lower=0)

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

# Keep eligible content and one row per content item
df["content_age_days"] = pd.to_numeric(
    df["content_age_days"], errors="coerce"
)
df = df[
    (df["impressions_90d"] > 0) &
    (df["content_age_days"] >= 90)
].drop_duplicates(subset=["content_id"]).reset_index(drop=True)

# Define the target separately; never use it as a feature
y = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)

X = df[numeric_features + categorical_features].copy()
X[numeric_features] = X[numeric_features].replace(
    [np.inf, -np.inf], np.nan
)

# Numeric missing values are filled with medians; categories use most frequent
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Fit preprocessing for this feature audit
X_vector = preprocessor.fit_transform(X)

print("Rows prepared:", len(X))
print("Original numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Transformed feature-vector shape:", X_vector.shape)
print("Target positive rate:", round(y.mean(), 4))
print("Missing target values:", int(y.isna().sum()))
print("Target excluded from X:", "trend_direction" not in X.columns)
print("Feature vector built successfully.")



Rows prepared: 30000
Original numeric features: 18
Categorical features: 8
Transformed feature-vector shape: (30000, 52)
Target positive rate: 0.5421
Missing target values: 0
Target excluded from X: True
Feature vector built successfully.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Numeric features

- `search_volume`, `competition`, `cpc`: search demand, competition, and cost-per-click indicators. Numeric conversion is applied; missing values are imputed with the training-set median during model preprocessing.
- `word_count`, `char_count`: content length measures. Missing values are median-imputed.
- `log_impressions_90d`, `log_clicks_90d`, `log_sessions_90d`, `log_ai_sessions_90d`: log-transformed performance counts over 90 days, reducing the influence of very large values. The raw counts are converted to numeric values, missing counts are filled with zero, and `log1p` is applied.
- `days_with_impressions`, `days_with_sessions`: number of days with measurable activity. Missing values are median-imputed.
- `content_age_days`, `days_since_last_update`: content age and time since the last update. Missing values are median-imputed.
- `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`: performance and engagement indicators. Missing values are median-imputed; infinite values are treated as missing.

### Categorical features

- `competition_level`: category describing competition.
- `content_type`: type of content.
- `main_intent`: intended search purpose.
- `age_tier`: content-age category.
- `freshness_tier`: content freshness category.
- `word_count_tier`: content-length category based on word count.
- `impression_tier`: category representing impressions.
- `position_tier`: category representing search position.

Missing categorical values are filled with `unknown` before encoding. One-hot encoding converts categories into numeric columns, and unseen categories are ignored by the encoder.

### Availability and timing

These features should only be used if their values are known at the time a prediction is made. The 90-day metrics and derived tiers must refer to a completed historical window, not a window containing future outcomes. The dataset's exact prediction cutoff must be verified before claiming that every feature is available beforehand.

The target is derived from `trend_direction == "down"`. `trend_direction` is excluded from the feature vector because it directly defines the target. The related `trend_pct` field is also excluded because it can reveal the outcome.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Check the feature definitions and missing-value handling

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nMissing values before preprocessing:")
print(X[numeric_features + categorical_features].isna().sum().sort_values(ascending=False).head(10))

print("\nFeature availability review:")
availability = pd.DataFrame({
    "feature": numeric_features + categorical_features,
    "type": (
        ["numeric"] * len(numeric_features)
        + ["categorical"] * len(categorical_features)
    ),
    "missing_before_preprocessing": [
        int(X[col].isna().sum())
        for col in numeric_features + categorical_features
    ],
    "included_in_X": [
        col in X.columns
        for col in numeric_features + categorical_features
    ]
})
display(availability)

print("\nLeakage safeguards:")
print("- Target is stored separately in y:", "trend_direction" not in X.columns)
print("- trend_pct is excluded:", "trend_pct" not in X.columns)
print("- Client and content IDs are excluded:",
      "client_id" not in X.columns and "content_id" not in X.columns)
print("- Numeric missing values are imputed in the preprocessing pipeline.")
print("- Categorical values are encoded in the preprocessing pipeline.")


Numeric features: 18
Categorical features: 8

Missing values before preprocessing:
char_count             7699
word_count             7699
competition            2468
search_volume          2468
cpc                    2468
scroll_rate             125
log_clicks_90d            0
log_sessions_90d          0
log_ai_sessions_90d       0
log_impressions_90d       0
dtype: int64

Feature availability review:


,feature,type,missing_before_preprocessing,included_in_X
0,search_volume,numeric,2468,True
1,competition,numeric,2468,True
2,cpc,numeric,2468,True
3,word_count,numeric,7699,True
4,char_count,numeric,7699,True
5,log_impressions_90d,numeric,0,True
6,log_clicks_90d,numeric,0,True
7,log_sessions_90d,numeric,0,True
8,log_ai_sessions_90d,numeric,0,True
9,days_with_impressions,numeric,0,True



Leakage safeguards:
- Target is stored separately in y: True
- trend_pct is excluded: True
- Client and content IDs are excluded: True
- Numeric missing values are imputed in the preprocessing pipeline.
- Categorical values are encoded in the preprocessing pipeline.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage checks

The target is `is_declining_label`, defined as 1 when `trend_direction` equals `"down"` and 0 otherwise.

- **Direct target leakage:** `trend_direction` defines the target, so it must not be included in the feature vector.
- **Related label leakage:** `trend_pct` is excluded because it describes the trend used to define the target.
- **Identifier leakage:** `client_id` and `content_id` are excluded from model inputs. They may be useful for grouping or tracking records, but should not be treated as predictive features.
- **Future-window leakage:** the 90-day metrics and derived features must be computed only from information available before the prediction cutoff. The dataset's exact timing must be verified before confirming this.
- **Preprocessing leakage:** imputers, scalers, and encoders must be fitted on training data only during model evaluation, then applied to held-out data.
- **Product and provider fields:** fields such as `provider_used` and `model_used` are excluded from the model feature list. Their availability and possible relationship to the outcome should be reviewed before considering them.

The checks below verify that known target-derived fields and identifiers are absent from the feature matrix. These checks reduce specific leakage risks but do not prove that every feature is temporally safe.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Test for known sources of feature leakage

forbidden_features = {
    "trend_direction",  # Directly defines the target
    "trend_pct",        # Related trend/outcome field
    "is_declining_label",# Target itself
    "client_id",        # Identifier, not a model feature
    "content_id"        # Identifier, not a model feature
}

actual_features = set(X.columns)
leaked_features = actual_features.intersection(forbidden_features)

print("Forbidden fields found in X:", sorted(leaked_features))
assert not leaked_features, (
    f"Potential leakage: forbidden fields found in X: {leaked_features}"
)

# Confirm that the target is separate from the feature matrix
assert len(X) == len(y), "Feature and target row counts do not match."
assert "trend_direction" not in X.columns
assert "trend_pct" not in X.columns
assert "is_declining_label" not in X.columns

# Check the preprocessing pipeline structure
assert "imputer" in numeric_pipeline.named_steps
assert "scaler" in numeric_pipeline.named_steps
assert "imputer" in categorical_pipeline.named_steps
assert "encoder" in categorical_pipeline.named_steps

print("PASS: known forbidden fields are excluded.")
print("PASS: feature and target row counts match.")
print("PASS: preprocessing includes imputation and encoding.")
print(
    "NOTE: future-window safety still requires verifying the "
    "feature timestamps against the prediction cutoff."
)


Forbidden fields found in X: []
PASS: known forbidden fields are excluded.
PASS: feature and target row counts match.
PASS: preprocessing includes imputation and encoding.
NOTE: future-window safety still requires verifying the feature timestamps against the prediction cutoff.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded fields and reasons

- `trend_direction`: excluded because it directly defines the target label.
- `trend_pct`: excluded because it is related to the trend outcome and could reveal information about the target.
- `is_declining_label`: excluded from the features because it is the target being predicted.
- `client_id`: excluded because it is an identifier, not a generalizable model feature. It may be used for client-level grouping during evaluation.
- `content_id`: excluded because it identifies individual content items rather than describing their characteristics.
- `provider_used`: excluded because it is not part of the approved model feature list and may reflect tooling or workflow choices.
- `model_used`: excluded because it is not part of the approved model feature list and may reflect tooling or workflow choices.
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`: excluded as separate raw features because they are not in the approved feature list. Their time windows would also need to be checked against the prediction cutoff before use.
- `pageviews_90d`, `users_90d`, `engaged_sessions_90d`, `scroll_events_90d`: excluded because they are not in the approved model feature list.
- `has_clicks`, `has_ai_sessions`, `measurable_opportunity`: excluded because these engineered flags are not in the approved model feature list.

The selected feature list follows `MODEL_NUMERIC_FEATURES` and `MODEL_CATEGORICAL_FEATURES` from `scripts/ml_utils.py`. Exclusion reduces known leakage risks, but does not by itself prove that all selected features are available before prediction.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Verify that excluded fields are not used as model features

excluded_fields = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "client_id",
    "content_id",
    "provider_used",
    "model_used",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "pageviews_90d",
    "users_90d",
    "engaged_sessions_90d",
    "scroll_events_90d",
    "has_clicks",
    "has_ai_sessions",
    "measurable_opportunity"
]

included_exclusions = [
    col for col in excluded_fields if col in X.columns
]

print("Excluded fields checked:", len(excluded_fields))
print("Excluded fields accidentally included:", included_exclusions)

assert not included_exclusions, (
    f"Some excluded fields are in X: {included_exclusions}"
)

# Confirm that X contains only the approved feature list
approved_features = set(numeric_features + categorical_features)
assert set(X.columns) == approved_features

print("PASS: excluded fields are absent from X.")
print("PASS: X contains only the approved feature list.")
print("Final feature count:", X.shape[1])


Excluded fields checked: 20
Excluded fields accidentally included: []
PASS: excluded fields are absent from X.
PASS: X contains only the approved feature list.
Final feature count: 26


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.